# SmolLM2-135M → AM-CeNN v2 + 8-Shard Top-2 FFN

This v2 notebook fixes the failure mode of the first all-at-once conversion.

**What changed**
- AM feature dimension increased from **32 → 128**;
- positive random features use antithetic `(ω, -ω)` pairs;
- every layer gets a zero-initialized trainable feature correction `ΔΩ`;
- attention is replaced **5 layers at a time**;
- each new AM-CeNN block is directly distilled against the original softmax-attention output on the **same teacher hidden state**;
- only after all 30 layers are calibrated do we run end-to-end CE + logit KL + hidden-state alignment;
- the 1536-wide FFN remains an exact 8×192 partition with Top-2 correction and `route_mix=0` warm start;
- no held-out benchmark is run.


In [ ]:
import subprocess, sys, pathlib, importlib, json, torch
subprocess.run(['nvidia-smi'], check=False)
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/vtavakkoli/TinyCeNN-LM.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), 'huggingface_hub'], check=True)
SRC = REPO_DIR / 'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
import tinycenn_lm
print('TinyCeNN import:', tinycenn_lm.__file__)


## Hugging Face login
Store a **new write token** in Colab Secrets as `HF_TOKEN`. Do not paste the token into the notebook.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download
HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets first.')
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()['name']
print('HF user:', HF_USER)


## Settings


In [ ]:
BASE_MODEL = 'HuggingFaceTB/SmolLM2-135M'
TARGET_REPO = f'{HF_USER}/SmolLM2-135M-AMCeNN-Top2-v2'
OUTPUT_DIR = REPO_DIR / 'checkpoints' / 'smollm2-amcenn-top2-v2'
CONTEXT_LENGTH = 128
FEATURE_DIM = 128
GROUP_SIZE = 5
CALIBRATION_STEPS = 30
FINAL_MAX_TOKENS = 500_000
MAX_RUNTIME_MINUTES = 60
print('base:', BASE_MODEL)
print('target:', TARGET_REPO)


## Train v2
Phase 1 performs direct layerwise attention-output calibration. Phase 2 performs a shorter global CE + KL + hidden-state distillation.


In [ ]:
cmd = [
    sys.executable, str(REPO_DIR / 'scripts' / 'train_smollm2_amcenn_v2.py'),
    '--base-model', BASE_MODEL,
    '--output-dir', str(OUTPUT_DIR),
    '--context-length', str(CONTEXT_LENGTH),
    '--feature-dim', str(FEATURE_DIM),
    '--group-size', str(GROUP_SIZE),
    '--calibration-steps', str(CALIBRATION_STEPS),
    '--final-max-tokens', str(FINAL_MAX_TOKENS),
    '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## Inspect calibration and final training report


In [ ]:
report = json.loads((OUTPUT_DIR / 'smollm2_amcenn_v2_training_report.json').read_text())
print(json.dumps({k: v for k, v in report.items() if k != 'calibration_stages'}, indent=2))
print('\nLayerwise calibration:')
print(f"{'layers':>9} | {'steps':>5} | {'first':>10} | {'last':>10} | {'NMSE':>9} | {'cos dist':>9}")
print('-' * 67)
for s in report['calibration_stages']:
    layers = f"{s['layers'][0]}-{s['layers'][-1]}"
    first = s['first_alignment_loss']
    first_s = 'n/a' if first is None else f'{first:.4f}'
    print(f"{layers:>9} | {s['steps']:5d} | {first_s:>10} | {s['last_alignment_loss']:10.4f} | {s['last_nmse']:9.4f} | {s['last_cosine_distance']:9.4f}")


## Publish as a separate v2 model


In [ ]:
api.create_repo(TARGET_REPO, repo_type='model', exist_ok=True)
api.upload_folder(
    repo_id=TARGET_REPO,
    repo_type='model',
    folder_path=str(OUTPUT_DIR),
    commit_message='Publish SmolLM2 AM-CeNN Top-2 v2 progressive student',
)
print('Published:', f'https://huggingface.co/{TARGET_REPO}')


## Reload and verify the final architecture


In [ ]:
from transformers import AutoTokenizer
from tinycenn_lm.smollm2_amcenn import ShardedTop2LlamaMLP
from tinycenn_lm.smollm2_amcenn_v2 import AMCeNNAttentionV2, build_smollm2_amcenn_v2
REMOTE_DIR = pathlib.Path(snapshot_download(TARGET_REPO, token=HF_TOKEN))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = (
    torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type == 'cuda' else torch.float32)
)
model = build_smollm2_amcenn_v2(REMOTE_DIR, device=device, dtype=dtype)
tokenizer = AutoTokenizer.from_pretrained(REMOTE_DIR, use_fast=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
am_layers = sum(isinstance(m, AMCeNNAttentionV2) for m in model.modules())
moe_layers = sum(isinstance(m, ShardedTop2LlamaMLP) for m in model.modules())
transformer_attention = [m.__class__.__name__ for m in model.modules() if 'LlamaAttention' in m.__class__.__name__]
print('AM-CeNN v2 layers:', am_layers)
print('8-shard Top-2 FFNs:', moe_layers)
print('Transformer self-attention remaining:', transformer_attention)
assert am_layers == 30 and moe_layers == 30 and not transformer_attention
print('STRUCTURE: PASS')


## Generation smoke test


In [ ]:
prompts = [
    'The capital of Austria is',
    'Artificial intelligence can help',
    'Once upon a time, a small robot was lost in a park.',
    'A small language model can',
]
model.eval()
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=80, min_new_tokens=20, do_sample=True,
            temperature=0.75, top_p=0.90, top_k=40, repetition_penalty=1.08,
            no_repeat_ngram_size=4, use_cache=False, pad_token_id=tokenizer.eos_token_id,
        )
    print('\n' + '=' * 90)
    print(tokenizer.decode(output[0], skip_special_tokens=True))
